In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from numpy.random import randint, rand
import matplotlib.pyplot as plt

/usr/local/lib/python3.10/dist-packages/yfinance/base.py:48: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  _empty_series = pd.Series()


In [2]:
def calculate_metrics(symbol, start_date, end_date, risk_free_rate, market_returns):
    # Download historical data for the stock
    stock_data = yf.download(symbol, start=start_date, end=end_date)

    # Extract relevant columns
    stock_returns = stock_data['Adj Close'].pct_change().dropna()

    # Calculate average return
    average_return = np.mean(stock_returns)

    # Calculate beta (sensitivity to market movements)
    cov_matrix = np.cov(stock_returns, market_returns)
    beta = cov_matrix[0, 1] / np.var(market_returns)

    # Calculate Sharpe Ratio
    sharpe_ratio = (average_return - risk_free_rate) / np.std(stock_returns)

    # Calculate Jensen's Alpha
    expected_return = risk_free_rate + beta * (np.mean(market_returns) - risk_free_rate)
    jensens_alpha = average_return - expected_return

    # Calculate Treynor Ratio
    treynor_ratio = (average_return - risk_free_rate) / beta

    return [symbol, sharpe_ratio, treynor_ratio, jensens_alpha]

In [3]:
# Define a list of stock symbols for 10 companies
companies = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "TSLA", "JNJ", "PG", "KO", "INTC", "IBM"
]

# Define the common date range for historical data
start_date = '2022-01-01'
end_date = '2023-01-01'

# Download historical data for the market index (S&P 500 as a proxy)
market_data = yf.download('^GSPC', start=start_date, end=end_date)
market_returns = market_data['Adj Close'].pct_change().dropna()

# Assume a risk-free rate (replace with the current risk-free rate)
risk_free_rate = 0.01

# Calculate metrics for each company
companies_data = [calculate_metrics(symbol, start_date, end_date, risk_free_rate, market_returns) for symbol in
                  companies]

# Display the results
columns = ["Company", "Sharpe Ratio", "Treynor Ratio", "Jensen's Alpha"]
df = pd.DataFrame(companies_data, columns=columns)
print(df)


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

  Company  Sharpe Ratio  Treynor Ratio  Jensen's Alpha
0    AAPL     -0.493773      -0.008446        0.003052
1    MSFT     -0.496693      -0.008567        0.002846
2    AMZN     -0.391302      -0.007493        0.005398
3   GOOGL     -0.480085      -0.008666        0.002843
4    TSLA     -0.337143      -0.007943        0.004936
5     JNJ     -0.886403      -0.031703       -0.006416
6      PG     -0.730041      -0.021180       -0.004959
7      KO     -0.768264      -0.019392       -0.004233
8    INTC     -0.513968      -0.010323        0.000539
9     IBM     -0.637293      -0.017767       -0.003760


In [4]:
def maximum(func, max_func):
    if func < max_func:
        return func
    else:
        return 0

In [5]:
def calculate_score(sharpe_ratio, treynor_ratio, jensens_alpha, max_sharpe=2, max_treynor=2, max_jensens=1):
    score = maximum(sharpe_ratio, max_sharpe) + maximum(jensens_alpha, max_jensens) + maximum(treynor_ratio, max_treynor)
    return score

In [6]:
def fittest(pop, fitness_scores, gen):
    fittest_index = fitness_scores.index(max(fitness_scores))
    fittest_portfolio = pop[fittest_index]
    print(gen, fittest_portfolio, "Fitness:", max(fitness_scores))

In [7]:
def selection(pop, scores, k=3):
    selection_ix = randint(len(pop))
    for ix in randint(0, len(pop), k - 1):
        if scores[ix] > scores[selection_ix]:
            selection_ix = ix
    return pop[selection_ix]

In [8]:
def crossover(p1, p2, r_cross):
    c1, c2 = p1.copy(), p2.copy()

    if rand() < r_cross:
        pt = randint(1, len(p1) - 2)
        c1 = p1[:pt] + p2[pt:]
        c2 = p2[:pt] + p1[pt:]

    return [c1, c2]


In [9]:
def mutation(bitstring, r_mut):
    for i in range(len(bitstring)):
        if rand() < r_mut:
            bitstring[i] = 1 - bitstring[i]
    return bitstring

In [10]:
def fitness_with_constant_ticket_size(portfolio, data, n, ticket_size=10000):
    total = 0
    final_score = 0

    for ele in range(len(portfolio)):
        total = sum(portfolio)

        if total == n:
            returns = [portfolio[ele] * data[ele][4] for ele in range(len(portfolio))]
            normalized_returns = sum(returns) / len(portfolio)
            final_score = normalized_returns / ticket_size

        else:
            final_score = -1000

    return final_score

In [11]:
num = 10
n_pop = 100
m = 4
r_cross = 0.95
r_mut = 0.05
n_gen = 500

In [12]:
scores = [calculate_score(*company[1:4]) for company in companies_data]
for i in range(len(companies_data)):
    companies_data[i].append(scores[i])
columns = ["Company", "Sharpe Ratio", "Treynor Ratio", "Jensen's Alpha", "Scores"]
df = pd.DataFrame(companies_data, columns=columns)
print(df)

  Company  Sharpe Ratio  Treynor Ratio  Jensen's Alpha    Scores
0    AAPL     -0.493773      -0.008446        0.003052 -0.499167
1    MSFT     -0.496693      -0.008567        0.002846 -0.502414
2    AMZN     -0.391302      -0.007493        0.005398 -0.393397
3   GOOGL     -0.480085      -0.008666        0.002843 -0.485909
4    TSLA     -0.337143      -0.007943        0.004936 -0.340149
5     JNJ     -0.886403      -0.031703       -0.006416 -0.924522
6      PG     -0.730041      -0.021180       -0.004959 -0.756180
7      KO     -0.768264      -0.019392       -0.004233 -0.791889
8    INTC     -0.513968      -0.010323        0.000539 -0.523752
9     IBM     -0.637293      -0.017767       -0.003760 -0.658820


In [ ]:
pop = [randint(0, 2, num).tolist() for _ in range(n_pop)]
fitt = []
for generation in range(n_gen):
    fitness_scores = [fitness_with_constant_ticket_size(individual, companies_data, m) for individual in pop]
    fitt.append(max(fitness_scores))
    fittest(pop, fitness_scores, generation)

    selected = [selection(pop, fitness_scores) for _ in range(n_pop)]
    children = []

    for i in range(0, n_pop, 2):
        p1, p2 = selected[i], selected[i + 1]

        for c in crossover(p1, p2, r_cross):
            mutation(c, r_mut)
            children.append(c)

    pop = children.copy()

0 [1, 1, 1, 1, 0, 0, 0, 0, 0, 0] Fitness: -1.8808879158906953e-05
1 [0, 1, 1, 0, 1, 0, 0, 0, 1, 0] Fitness: -1.7597131031193865e-05
2 [0, 1, 1, 0, 1, 0, 0, 0, 1, 0] Fitness: -1.7597131031193865e-05
3 [1, 0, 0, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.848978138065163e-05
4 [1, 0, 0, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.848978138065163e-05
5 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
6 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
7 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
8 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
9 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
10 [0, 1, 1, 1, 1, 0, 0, 0, 0, 0] Fitness: -1.7218698013304675e-05
11 [1, 0, 1, 1, 1, 0, 0, 0, 0, 0] Fitness: -1.7186231753601377e-05
12 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
13 [0, 0, 1, 1, 1, 0, 0, 0, 1, 0] Fitness: -1.743208154400044e-05
14 [1, 0, 1, 1, 1, 0, 0, 0, 0, 0] Fitness: -1.7186231753601377e-05
15 [1, 0, 1, 1

In [ ]:
plt.plot(fitt)
plt.xlabel('Generations')
plt.ylabel('Fittest in Population')
plt.title('Fitness Convergence')
plt.show()